# Exercise 00
## PyTorch Introduction - Part I: Tensors

PyTorch is a scientific computing package with two main purposes:

1. A NumPy-like library for n-dimensional arrays (*tensors*) that can use the power of GPUs.
2. A deep learning framework with automatic differentiation to build and train neural networks.

This notebook covers the first part, the next notebook the second.

### Goals of this notebook

- Know the similarities and differences between NumPy arrays and PyTorch tensors.
- Manipulate tensors: indexing, reshaping, math, reductions and broadcasting.
- **Write vectorized code**: replace slow Python loops with tensor operations and existing library functions.
- Solve your first small tasks in `exercise_code` and learn how the exercises work.

## Setup

Every exercise notebook starts with similar setup cells. On Google Colab, the first cell mounts your Google Drive and changes into the exercise folder. We presume that you uploaded the `cv3dst` folder to the top level of your drive. Locally, it only imports the packages.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

USING_COLAB = "google.colab" in sys.modules

if USING_COLAB:
    import os
    from google.colab import drive

    gdrive_path = "/content/gdrive/MyDrive/cv3dst/exercise_00"

    # This will mount your google drive under 'MyDrive'
    drive.mount("/content/gdrive", force_remount=True)
    # Navigate to the exercise folder and check manually that all files are present
    os.chdir(gdrive_path)
    print(sorted(os.listdir()))

# shorter and more readable printing of arrays and tensors
torch.set_printoptions(precision=3, sci_mode=False)
np.set_printoptions(precision=3, suppress=True)

print(f"PyTorch {torch.__version__}, NumPy {np.__version__}")

The tasks of this notebook are located in python files in the `exercise_code` folder (section 9). The `autoreload` extension reloads these files automatically when you change them, so you do not need to restart the notebook. It has to be loaded **before** the import.

In [ ]:
%load_ext autoreload
%autoreload 2

### A first taste: loops vs. vectorized code

The most important lesson of this notebook comes first. We compute the sum of squares of 100,000 numbers, once with a Python loop and once with a single tensor operation. `%timeit` runs a statement several times and reports the average run time.

In [ ]:
x = torch.rand(100_000)


def sum_of_squares_loop(x):
    total = 0.0
    for v in x:
        total += v * v
    return total


%timeit -n 1 -r 1 sum_of_squares_loop(x)
%timeit (x**2).sum()

The vectorized version is several orders of magnitude faster. In a loop, Python processes every element on its own: it creates objects, looks up the operation and checks types, once per element. A tensor operation does this once and then runs a compiled (and often parallel) kernel over all elements.

<div class="alert alert-block alert-info"><b>Rule of thumb:</b> whenever you are about to write a <code>for</code> loop over the elements of a tensor, stop and look for a vectorized formulation or an existing function.</div>

The next sections give you the tools to do this. Section 8 comes back to vectorization with more realistic examples, and in section 9 it is your turn.

## 1. Tensors and NumPy arrays

[`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) is the central class of PyTorch. Tensors are very similar to NumPy's `ndarray`, but in addition they

* can be moved to a GPU to accelerate computations,
* can track the operations applied to them in order to compute gradients (autograd).

Let us create a NumPy array and a tensor from the same data.

In [ ]:
data = [[1, 2, 3], [4, 5, 6]]
a_np = np.array(data)
a_ts = torch.tensor(data)

print(f"NumPy: {type(a_np)}, shape {a_np.shape}, dtype {a_np.dtype}")
print(a_np)
print(f"Torch: {type(a_ts)}, shape {a_ts.shape}, dtype {a_ts.dtype}, device {a_ts.device}")
print(a_ts)

Most creation functions exist in both libraries with the same name. Note the different default float types: NumPy uses `float64`, PyTorch uses `float32`.

In [ ]:
print(np.zeros((2, 3)).dtype, torch.zeros(2, 3).dtype)
print(np.arange(5), torch.arange(5))
print(np.linspace(0, 1, 5), torch.linspace(0, 1, 5))
print(np.random.rand(3), torch.rand(3))

### NumPy vs. PyTorch at a glance

The APIs are intentionally similar, so your NumPy knowledge transfers almost directly. These are the differences you will run into most often:

| | NumPy | PyTorch |
|---|---|---|
| Array type | `np.ndarray` | `torch.Tensor` |
| Default float type | `float64` | `float32` |
| Name of the axis argument | `axis=` | `dim=` (most functions also accept `axis=`) |
| Shape | `a.shape` | `t.shape` or `t.size()` |
| Independent copy | `a.copy()` | `t.clone()` |
| Change the type | `a.astype(np.float32)` | `t.float()`, `t.to(torch.float32)` |
| Concatenation | `np.concatenate` | `torch.cat` |
| In-place operations | `a += 1` | `t += 1` or methods with a trailing `_`, e.g. `t.add_(1)` |
| Devices | CPU only | CPU, GPU: `t.to("cuda")` |
| Gradients | - | `requires_grad=True` and `backward()` |
| Random seed | `np.random.seed(0)` | `torch.manual_seed(0)` (a separate generator!) |

## 2. Conversion between NumPy arrays and tensors

`torch.from_numpy` creates a tensor that **shares the memory** of the NumPy array, while `torch.tensor` **copies** the data. Changing the array therefore changes one tensor, but not the other.

In [ ]:
a_np = np.array([1.0, 2.0, 3.0])
shared = torch.from_numpy(a_np)  # shares memory with a_np
copied = torch.tensor(a_np)  # independent copy

a_np[0] = -1.0
print(f"numpy array: {a_np}")
print(f"from_numpy:  {shared}")
print(f"tensor:      {copied}")

Note that the tensors kept the `float64` type of the NumPy array.

The conversion back with `.numpy()` also shares memory. It only works for tensors on the CPU that do not track gradients (see the next notebook). The safe way to convert any tensor is `t.detach().cpu().numpy()`.

In [ ]:
t = torch.ones(3, requires_grad=True)
try:
    t.numpy()
except RuntimeError as e:
    print(f"RuntimeError: {e}")

print(t.detach().cpu().numpy())

## 3. Indexing

Tensors support the same indexing as NumPy arrays.

In [ ]:
t = torch.arange(12).reshape(3, 4)
print(t)
print(t[0])  # first row
print(t[:, 1])  # second column
print(t[1:, ::2])  # rows 1 and 2, every second column
print(t[-1, -1])  # last element
print(t[[0, 2]])  # rows 0 and 2 (indexing with a list of indices)

### Boolean masks

A comparison returns a boolean tensor of the same shape, a so-called *mask*. Indexing with a mask selects all elements where the mask is `True`. The result is always 1D. Masks replace many `if` statements inside loops.

In [ ]:
mask = t > 5
print(mask)
print(t[mask])

t_clipped = t.clone()
t_clipped[t_clipped > 5] = 5  # assign a value to all selected elements
print(t_clipped)

To check whether two tensors are identical, `==` is not enough: it compares element-wise and returns a tensor. `torch.equal` returns a single `bool`.

In [ ]:
print(t[mask] == t[t > 5])
print(torch.equal(t[mask], t[t > 5]))

### Indexing with index tensors

Instead of a list, you can also index with integer tensors. If you index several dimensions with index tensors, they are paired element by element: `t[rows, cols]` returns the elements `t[rows[0], cols[0]], t[rows[1], cols[1]], ...`. This reads, e.g., the values of a heatmap at the predicted positions of several objects in one step.

In [ ]:
heatmap = torch.arange(20).reshape(4, 5)
ys = torch.tensor([0, 2, 3])  # rows (y) of three objects
xs = torch.tensor([4, 1, 1])  # columns (x) of three objects
print(heatmap)
print(heatmap[ys, xs])  # heatmap[0, 4], heatmap[2, 1], heatmap[3, 1]

The index tensor can have any shape, and the result has the shape of the index tensor. A tensor of values indexed with a tensor of indices therefore works as a *lookup table*. A common example in segmentation: datasets store their own class ids, which are mapped to the ids used for training.

In [ ]:
dataset_to_train_id = torch.tensor([0, 0, 1, 1, 2])  # e.g. merge dataset classes 0 and 1, and 2 and 3
labels = torch.tensor([[4, 3, 3], [0, 1, 2]])  # a tiny label map with dataset ids
print(dataset_to_train_id[labels])  # same shape as labels

### Views vs. copies

Slicing does not copy data. It returns a **view**, a new tensor that shares its memory with the original. Modifying the view modifies the original! NumPy behaves the same way. Use `.clone()` (NumPy: `.copy()`) when you need an independent copy.

In [ ]:
t = torch.arange(6).reshape(2, 3)
view = t[:, :2]
view[:, 0] = 100
print(t)  # t has changed as well!

t = torch.arange(6).reshape(2, 3)
copy = t[:, :2].clone()
copy[:, 0] = 100
print(t)  # t is unchanged

<div class="alert alert-block alert-warning"><b>Caution:</b> bugs caused by views are hard to find. In contrast to slicing, indexing with a mask or a list of indices always returns a copy.</div>

## 4. Changing the shape

`reshape` changes the shape while keeping the number of elements. One dimension can be `-1`; its size is then inferred.

In [ ]:
t = torch.arange(24)
print(t.reshape(2, 3, 4).shape)
print(t.reshape(4, -1).shape)

`permute` reorders the dimensions. A common use case: images are stored as `H x W x C` (e.g. by NumPy, PIL, matplotlib), but PyTorch expects `C x H x W`.

In [ ]:
img = torch.rand(32, 48, 3)  # H x W x C
img_chw = img.permute(2, 0, 1)  # C x H x W
print(img.shape, img_chw.shape)

`unsqueeze(dim)` inserts a new dimension of size 1, `squeeze()` removes all dimensions of size 1. Indexing with `None` also inserts a dimension; this is often shorter and works in NumPy as well.

In [ ]:
v = torch.rand(3)
print(v.unsqueeze(0).shape, v[None, :].shape)  # (1, 3)
print(v.unsqueeze(1).shape, v[:, None].shape)  # (3, 1)
print(torch.rand(1, 3, 1).squeeze().shape)  # (3,)

`torch.cat` joins tensors along an **existing** dimension, `torch.stack` creates a **new** one. The NumPy equivalents are `np.concatenate` and `np.stack`.

In [ ]:
a, b = torch.zeros(2, 3), torch.ones(2, 3)
print(torch.cat([a, b], dim=0).shape)  # (4, 3)
print(torch.cat([a, b], dim=1).shape)  # (2, 6)
print(torch.stack([a, b], dim=0).shape)  # (2, 2, 3)

## 5. Element-wise math and data types

Arithmetic operators work element-wise, just like in NumPy. Each operator also has a function form, e.g. `x + y` is `torch.add(x, y)`.

In [ ]:
x = torch.tensor([[1, 2], [3, 4]])
y = torch.tensor([[5, 6], [7, 8]])

print(x + y)
print(x * y)  # element-wise product, NOT the matrix product
print(x**2)
print(torch.exp(x.float()))

`clamp` limits all values to a range. A typical use is to clip predicted boxes `(x1, y1, x2, y2)` to the image. `0::2` selects the x coordinates, `1::2` the y coordinates. torchvision already provides this as [`torchvision.ops.clip_boxes_to_image`](https://pytorch.org/vision/stable/generated/torchvision.ops.clip_boxes_to_image.html).

In [ ]:
import torchvision

H, W = 480, 640
boxes = torch.tensor([[-5.0, 10.0, 50.0, 520.0], [300.0, -2.0, 700.0, 40.0]])

clipped = boxes.clone()
clipped[:, 0::2] = clipped[:, 0::2].clamp(min=0, max=W)
clipped[:, 1::2] = clipped[:, 1::2].clamp(min=0, max=H)
print(clipped)
print(torch.equal(clipped, torchvision.ops.clip_boxes_to_image(boxes, (H, W))))

Dividing two integer tensors with `/` returns floats in both libraries. `//` is integer (floor) division.

In [ ]:
print(x / y)
print(x // y)
print(x.numpy() / y.numpy())
print(x.numpy() // y.numpy())

When types are mixed, the result gets the more general type (*type promotion*). Mixed types are a common source of errors, e.g. data loaded with NumPy is `float64`, while network weights are `float32`.

In [ ]:
ints = torch.tensor([1, 2])
print((ints + 0.5).dtype)  # int64 + python float -> float32
print((ints + torch.tensor([0.5], dtype=torch.float64)).dtype)  # -> float64

### Comparing floats

Floating-point arithmetic is not exact. Mathematically equivalent computations can differ in the last digits, e.g. because they sum in a different order. Never compare floats with `==`. Use `torch.isclose` (element-wise) or `torch.allclose` (single `bool`) instead; NumPy has `np.isclose` and `np.allclose`. The tests in all exercises use these functions.

In [ ]:
s = torch.tensor([0.1, 0.2], dtype=torch.float64).sum()
print(s.item())
print(s == 0.3)
print(torch.isclose(s, torch.tensor(0.3, dtype=torch.float64)))

## 6. Reductions

Reductions such as `sum`, `mean`, `max` and `argmax` combine the elements along a dimension. Think of `dim` as **the dimension that disappears**. `keepdim=True` keeps it with size 1, which is useful for broadcasting (section 7).

In [ ]:
t = torch.tensor([[1.0, 5.0, 2.0], [4.0, 3.0, 6.0]])  # (2, 3)

print(t.sum())  # all elements
print(t.sum(dim=0))  # dim 0 disappears: one value per column, (3,)
print(t.sum(dim=1))  # dim 1 disappears: one value per row, (2,)
print(t.mean(dim=1, keepdim=True).shape)  # (2, 1)

`max(dim=...)` returns the maximum values and their indices. If you only need the indices, use `argmax`. NumPy uses `axis` instead of `dim`.

In [ ]:
values, indices = t.max(dim=1)
print(values, indices)
print(t.argmax(dim=1))
print(np.argmax(t.numpy(), axis=1))

`sort` returns the sorted values and the order of the indices, `topk` only the `k` largest values. A detector, e.g., keeps only its most confident detections.

In [ ]:
scores = torch.tensor([0.3, 0.9, 0.1, 0.7])
values, order = scores.sort(descending=True)
print(values, order)
print(scores.topk(2))

`any` and `all` reduce boolean tensors: is at least one / is every element `True`? `nonzero` returns the indices of all `True` elements, one row per element. For masks, this answers questions such as "which rows of a segmentation mask contain the object?".

In [ ]:
mask = torch.tensor([[False, False, False], [False, True, True], [False, True, False]])
print(mask.any(dim=1))  # rows that contain the object
print(mask.any(dim=0))  # columns that contain the object
print(mask.all())
print(mask.nonzero())  # (y, x) of every pixel of the object

A reduction over all elements still returns a tensor. `.item()` converts a tensor with a single element to a python number, e.g. for printing or logging.

In [ ]:
total = t.sum()
print(total, type(total))
print(total.item(), type(total.item()))

## 7. Broadcasting

Broadcasting combines tensors of different shapes without copying data. It follows the same rules as in NumPy:

1. Compare the shapes starting from the **last** dimension. A missing dimension counts as size 1.
2. Two dimensions are compatible if they are equal or if one of them is 1.
3. Dimensions of size 1 are (virtually) stretched to the size of the other tensor.

| Shape A | Shape B | Result | Meaning |
|---|---|---|---|
| `(2, 3)` | `(3,)` | `(2, 3)` | add the vector to every row |
| `(2, 3)` | `(2, 1)` | `(2, 3)` | add the vector to every column |
| `(3, 1)` | `(1, 4)` | `(3, 4)` | all pairs of elements ("outer" operation) |
| `(2, 3)` | `(2,)` | error | last dimensions 3 and 2 do not match |

In [ ]:
m = torch.zeros(2, 3)
row = torch.tensor([1.0, 2.0, 3.0])  # (3,)
col = torch.tensor([10.0, 20.0])  # (2,)

print(m + row)  # (2, 3) + (3,): no unsqueeze needed
print(m + col[:, None])  # (2, 3) + (2, 1): the new dimension is needed

try:
    m + col  # (2, 3) + (2,)
except RuntimeError as e:
    print(f"RuntimeError: {e}")

The "outer" pattern with shapes `(N, 1)` and `(1, M)` creates an `N x M` result from all pairs of elements. You will use it all the time, e.g. to compare every detection with every track.

In [ ]:
a = torch.arange(3)
b = torch.arange(4)
print(a[:, None] + b[None, :])  # (3, 1) + (1, 4) -> (3, 4)

### Matrix multiplication

`@` (or `torch.matmul`) is the matrix product. It also broadcasts: all dimensions except the last two are treated as batch dimensions. This lets you multiply a whole batch of matrices without a loop.

In [ ]:
shape_pairs = [
    ((3,), (3,)),  # vector @ vector: dot product
    ((3, 4), (4,)),  # matrix @ vector
    ((3, 4), (4, 5)),  # matrix @ matrix
    ((10, 3, 4), (10, 4, 5)),  # batched matrix @ batched matrix
    ((10, 3, 4), (4, 5)),  # batched matrix @ broadcasted matrix
]
for s1, s2 in shape_pairs:
    out = torch.rand(s1) @ torch.rand(s2)
    print(f"{str(s1):>11} @ {str(s2):<11} -> {tuple(out.shape)}")

[`torch.einsum`](https://pytorch.org/docs/stable/generated/torch.einsum.html) expresses many linear algebra operations in a short notation based on the Einstein summation convention. Every dimension gets a letter; letters that do not appear in the output are summed over. It is especially handy for batched operations and products of more than two tensors.

We compare the results with `allclose`, because `einsum` and `@` may sum in a different order (see section 5).

In [ ]:
A = torch.rand(10, 3, 4)
B = torch.rand(10, 4, 5)
print(torch.allclose(torch.einsum("bij,bjk->bik", A, B), A @ B))

v1, M, v2 = torch.rand(3), torch.rand(3, 3), torch.rand(3)
print(torch.allclose(torch.einsum("i,ij,j->", v1, M, v2), v1 @ M @ v2))

## 8. Vectorization in practice

Now we have all the tools to replace loops. We use the following recipe:

1. Write down the naive loop to get the math right.
2. Replace the loop with broadcasting, masks and reductions.
3. Check whether PyTorch or torchvision already provides the function.
4. Check the result against the loop with `torch.allclose`.

### 8.1 Normalizing an image per channel

Neural networks usually expect images normalized with a per-channel mean and standard deviation.

In [ ]:
import torchvision.transforms.functional as TF

img = torch.rand(3, 64, 64)  # C x H x W
mean = torch.tensor([0.485, 0.456, 0.406])
std = torch.tensor([0.229, 0.224, 0.225])


def normalize_loop(img, mean, std):
    out = torch.empty_like(img)
    C, H, W = img.shape
    for c in range(C):
        for y in range(H):
            for x in range(W):
                out[c, y, x] = (img[c, y, x] - mean[c]) / std[c]
    return out


def normalize_vectorized(img, mean, std):
    # (C, H, W) - (C, 1, 1): mean and std are broadcast over all pixels
    return (img - mean[:, None, None]) / std[:, None, None]


out_loop = normalize_loop(img, mean, std)
print(torch.allclose(out_loop, normalize_vectorized(img, mean, std)))
print(torch.allclose(out_loop, TF.normalize(img, mean.tolist(), std.tolist())))

%timeit -n 1 -r 1 normalize_loop(img, mean, std)
%timeit normalize_vectorized(img, mean, std)
%timeit TF.normalize(img, mean.tolist(), std.tolist())

The vectorized function is written with operations that exist in NumPy as well, so it works unchanged on NumPy arrays:

In [ ]:
out_np = normalize_vectorized(img.numpy(), mean.numpy(), std.numpy())
print(type(out_np), np.allclose(out_np, out_loop.numpy()))

### 8.2 Replacing `if` statements: leaky ReLU

The leaky ReLU activation returns $x$ for positive inputs and $0.1 x$ otherwise. A condition inside a loop becomes a mask or [`torch.where`](https://pytorch.org/docs/stable/generated/torch.where.html), which picks elements from two tensors depending on a condition.

In [ ]:
import torch.nn.functional as F

x = torch.randn(10_000)


def leaky_relu_loop(x, slope=0.1):
    out = torch.empty_like(x)
    for i in range(len(x)):
        if x[i] > 0:
            out[i] = x[i]
        else:
            out[i] = slope * x[i]
    return out


def leaky_relu_vectorized(x, slope=0.1):
    return torch.where(x > 0, x, slope * x)


out_loop = leaky_relu_loop(x)
print(torch.allclose(out_loop, leaky_relu_vectorized(x)))
print(torch.allclose(out_loop, F.leaky_relu(x, negative_slope=0.1)))

%timeit -n 1 -r 1 leaky_relu_loop(x)
%timeit leaky_relu_vectorized(x)
%timeit F.leaky_relu(x, negative_slope=0.1)

### 8.3 All pairs: which points lie inside which boxes?

Given $N$ points and $M$ bounding boxes in the format `(x1, y1, x2, y2)`, we want an $N \times M$ boolean matrix that says whether point $i$ lies inside box $j$. The loop version needs two nested loops. The vectorized version uses the "outer" broadcasting pattern from section 7: points get the shape `(N, 1)`, boxes the shape `(1, M)`.

In [ ]:
N, M = 200, 50
points = torch.rand(N, 2) * 100
top_left = torch.rand(M, 2) * 80
boxes = torch.cat([top_left, top_left + 20], dim=1)  # (M, 4)


def points_in_boxes_loop(points, boxes):
    inside = torch.zeros(len(points), len(boxes), dtype=torch.bool)
    for i, (px, py) in enumerate(points):
        for j, (x1, y1, x2, y2) in enumerate(boxes):
            if x1 <= px <= x2 and y1 <= py <= y2:
                inside[i, j] = True
    return inside


def points_in_boxes_vectorized(points, boxes):
    px, py = points[:, 0, None], points[:, 1, None]  # (N, 1)
    x1, y1, x2, y2 = boxes[None, :, 0], boxes[None, :, 1], boxes[None, :, 2], boxes[None, :, 3]  # (1, M)
    return (x1 <= px) & (px <= x2) & (y1 <= py) & (py <= y2)  # (N, M)


inside = points_in_boxes_vectorized(points, boxes)
print(inside.shape, torch.equal(points_in_boxes_loop(points, boxes), inside))
print(f"points per box: {inside.sum(dim=0)[:10].tolist()} ...")

%timeit -n 1 -r 1 points_in_boxes_loop(points, boxes)
%timeit points_in_boxes_vectorized(points, boxes)

Note that `&` combines masks element-wise; python's `and` does not work on tensors.

<div class="alert alert-block alert-warning"><b>Memory:</b> vectorization trades memory for speed. An <code>(N, M, D)</code> intermediate tensor can become very large. If it does not fit into memory, loop over chunks of the data, not over single elements.</div>

### 8.4 Counting: a histogram of class labels

Many common operations already exist as library functions. Counting how often each class occurs is one call to `torch.bincount`.

In [ ]:
labels = torch.randint(0, 10, (10_000,))


def count_loop(labels, num_classes=10):
    counts = torch.zeros(num_classes, dtype=torch.long)
    for label in labels:
        counts[label] += 1
    return counts


def count_vectorized(labels, num_classes=10):
    # (N, 1) == (1, C) -> (N, C) one-hot matrix, then sum over all samples
    return (labels[:, None] == torch.arange(num_classes)[None, :]).sum(dim=0)


counts = count_loop(labels)
print(counts)
print(torch.equal(counts, count_vectorized(labels)))
print(torch.equal(counts, torch.bincount(labels, minlength=10)))

%timeit -n 1 -r 1 count_loop(labels)
%timeit count_vectorized(labels)
%timeit torch.bincount(labels, minlength=10)

<div class="alert alert-block alert-info">
<b>Rules of thumb for fast code</b>
<ul>
<li>Search the documentation of PyTorch and torchvision before you implement something yourself. Library functions are fast and tested.</li>
<li>Avoid loops over the elements of a tensor. Loop only over things that cannot be batched, e.g. epochs or files.</li>
<li>Replace <code>if</code> statements with masks, <code>torch.where</code> or <code>clamp</code>.</li>
<li>For "all pairs" problems, use broadcasting with shapes <code>(N, 1, ...)</code> and <code>(1, M, ...)</code>.</li>
<li>Keep a simple reference implementation and compare with <code>torch.allclose</code>.</li>
</ul>
</div>

## 9. Your turn: the first tasks

From now on, you solve the tasks of every exercise in the python files of the `exercise_code` folder. The notebooks import your code, run it and test it. At the end of exercise 0 (in notebook 3), you zip the folder and upload it to our test server, which grades it.

<div class="alert alert-info">
    <h3>Task: Ice-blue cells</h3>
    <p>These <b>ice-blue</b> cells mark tasks that you need to complete. The tasks of this notebook are located in <a href="exercise_code/tensor_ops.py"><code>exercise_code/tensor_ops.py</code></a>. Write your code between the <code>TODO</code> and <code>END OF YOUR CODE</code> comments.</p>
</div>

<div class="alert alert-danger">
    <h3>Test: Pink cells</h3>
    <p>The <b>pink</b> cells run tests for your implementation. The test server runs the same checks with other random inputs, so if all tests here pass, your code will most likely also pass on the server. Every task gives a small number of points.</p>
</div>

The tests only check that your results are correct. Still, follow the recipe of section 8: solutions **without loops** over the elements are highly encouraged. Look for existing PyTorch functions first. After every test, we compare the speed of your function with the library function, if there is one. Remember from section 8 how much slower a loop is.

In [ ]:
from exercise_code.tensor_ops import colorize_labels, select_scores, stable_softmax
from exercise_code.test import test_colorize_labels, test_select_scores, test_stable_softmax

### 9.1 Selecting the score of the correct class

A classifier outputs a score for every class, i.e. a tensor `scores` of shape `(N, C)` for $N$ samples and $C$ classes. Many losses, such as the cross-entropy loss, need the score of the **correct** class of every sample: `scores[i, labels[i]]`.

<div class="alert alert-info">
    <h3>Task: <code>select_scores</code></h3>
    <p>Implement <code>select_scores</code> in <a href="exercise_code/tensor_ops.py"><code>exercise_code/tensor_ops.py</code></a> without a loop. Hint: section 3 shows how to index with index tensors. PyTorch also provides a function for exactly this.</p>
</div>

<div class="alert alert-danger">
    <h3>Test: <code>select_scores</code></h3>
    <p>Run the following cells to test your implementation and to compare its speed.</p>
</div>

In [ ]:
_ = test_select_scores()

In [ ]:
scores, labels = torch.randn(10_000, 80), torch.randint(0, 80, (10_000,))
print(torch.equal(select_scores(scores, labels), torch.gather(scores, 1, labels[:, None])[:, 0]))
%timeit select_scores(scores, labels)
%timeit torch.gather(scores, 1, labels[:, None])

### 9.2 A numerically stable softmax

The softmax converts the scores $x$ of a classifier into probabilities:

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Computed naively, $e^{x}$ overflows to `inf` for scores above ~88 in `float32`, and the result becomes `nan`. Subtracting the maximum score $m$ from all scores solves this, because $\frac{e^{x_i - m}}{\sum_j e^{x_j - m}} = \frac{e^{-m} e^{x_i}}{e^{-m} \sum_j e^{x_j}}$ gives the same result, and the largest exponent is now $e^0 = 1$.

In [ ]:
x = torch.tensor([10.0, 100.0, 1000.0])
print(x.exp() / x.exp().sum())

<div class="alert alert-info">
    <h3>Task: <code>stable_softmax</code></h3>
    <p>Implement <code>stable_softmax</code> in <a href="exercise_code/tensor_ops.py"><code>exercise_code/tensor_ops.py</code></a> with basic tensor operations: here the point is to understand the numerical trick. The function must work for inputs of any shape along any dimension <code>dim</code>. Hint: <code>keepdim=True</code> from section 6.</p>
</div>

<div class="alert alert-danger">
    <h3>Test: <code>stable_softmax</code></h3>
    <p>Run the following cells to test your implementation and to compare it with <code>torch.softmax</code>.</p>
</div>

In [ ]:
_ = test_stable_softmax()

In [ ]:
logits = 100 * torch.randn(10_000, 80)
print(stable_softmax(x, dim=0))
print(torch.allclose(stable_softmax(logits, dim=1), torch.softmax(logits, dim=1)))
%timeit stable_softmax(logits, dim=1)
%timeit torch.softmax(logits, dim=1)

### 9.3 Visualizing a segmentation

A semantic segmentation assigns a class to every pixel. The result is a *label map* of shape `(H, W)` that contains the class index of every pixel. To look at it, we give every class a color from a *palette* of shape `(C, 3)`.

We create a small synthetic scene with three objects: a circle, a rectangle and a triangle. Every object is a boolean mask of shape `(H, W)`, computed with broadcasting from the pixel coordinates. The label map is 0 for the background and $k$ for the pixels of object $k$.

In [ ]:
H, W = 120, 160
ys, xs = torch.arange(H)[:, None], torch.arange(W)[None, :]  # (H, 1) and (1, W)
circle = (xs - 40) ** 2 + (ys - 60) ** 2 < 25**2
rectangle = (xs >= 90) & (xs < 145) & (ys >= 15) & (ys < 60)
triangle = (ys >= 70) & (ys < 110) & ((xs - 115).abs() <= (ys - 70) // 2)
masks = torch.stack([circle, rectangle, triangle])  # (3, H, W)

label_map = (masks * torch.arange(1, 4)[:, None, None]).amax(dim=0)  # (H, W): 0 = background
palette = torch.tensor([[0, 0, 0], [230, 25, 75], [60, 180, 75], [0, 130, 200]], dtype=torch.uint8)
print(masks.shape, label_map.shape, label_map.unique())

<div class="alert alert-info">
    <h3>Task: <code>colorize_labels</code></h3>
    <p>Implement <code>colorize_labels</code> in <a href="exercise_code/tensor_ops.py"><code>exercise_code/tensor_ops.py</code></a>. It returns the color of every pixel, i.e. a tensor of shape <code>(H, W, 3)</code> with the dtype of the palette. It should also work for a batch of label maps of shape <code>(B, H, W)</code>. Hint: section 3.</p>
</div>

<div class="alert alert-danger">
    <h3>Test: <code>colorize_labels</code></h3>
    <p>Run the following cells to test your implementation and to look at the scene.</p>
</div>

In [ ]:
_ = test_colorize_labels()

In [ ]:
colors = colorize_labels(label_map, palette)
big_label_map = torch.randint(0, 4, (1080, 1920))  # a Full HD label map
%timeit colorize_labels(big_label_map, palette)

plt.imshow(colors)  # matplotlib expects images as H x W x 3
plt.axis("off")
plt.show()

## 10. Practical tips

A few things that come up again and again when working with images, masks and tracks.

### Images and masks are `uint8`

Images and segmentation masks are usually stored and loaded as `uint8`, i.e. integers from 0 to 255. Arithmetic on `uint8` silently **wraps around**. Convert video frames to `float` before you compute anything with them, usually scaled to `[0, 1]`. Segmentation masks used as class indices, e.g. as targets of the cross-entropy loss, must be converted to `long`.

In [ ]:
import torch.nn.functional as F

frame = torch.tensor([[250, 100], [30, 200]], dtype=torch.uint8)  # e.g. a video frame loaded from disk
print(frame + 10)  # 250 + 10 wraps around to 4!
print(frame.float() / 255)

mask = torch.tensor([[[0, 1], [2, 1]]], dtype=torch.uint8)  # a segmentation mask (1, H, W) loaded from disk
logits = torch.randn(1, 3, 2, 2)  # (B, C, H, W) prediction for 3 classes
try:
    F.cross_entropy(logits, mask)
except RuntimeError as e:
    print(f"RuntimeError: {e}")
print(F.cross_entropy(logits, mask.long()))

### Feature maps and per-pixel features

Networks output feature maps of shape `(B, C, H, W)`. To classify or cluster the pixels, we often need one feature vector per pixel, i.e. the shape `(B, H·W, C)`, or `(B·H·W, C)` for all pixels of the batch.

`view` only works if the new shape is compatible with the memory layout of the tensor. After `permute` or `transpose`, the tensor is no longer *contiguous* in memory, and `view` fails. `reshape` works in both cases: it returns a view if possible and a copy otherwise.

In [ ]:
feats = torch.randn(2, 64, 30, 40)  # (B, C, H, W), e.g. the output of a backbone
pixels = feats.flatten(2).transpose(1, 2)  # (B, H*W, C): one feature vector per pixel
print(pixels.shape, pixels.is_contiguous())

back = pixels.transpose(1, 2).reshape(2, 64, 30, 40)
print(torch.equal(back, feats))

all_pixels = feats.permute(0, 2, 3, 1)  # (B, H, W, C), not contiguous
try:
    all_pixels.view(-1, 64)
except RuntimeError as e:
    print(f"RuntimeError: {e}")
print(all_pixels.reshape(-1, 64).shape)  # (B*H*W, C)

### Pairwise tensors: `expand` vs. `repeat`

Broadcasting never copies data. Sometimes you need the full pairwise tensor explicitly, e.g. one feature vector for every pair of a track and a detection, of shape `(N, M, D)`. `expand` creates it as a view without copying: the new dimension gets a *stride* of 0, i.e. all entries along it point to the same memory. `repeat` copies the data. For 500 tracks, 400 detections and 128-dimensional features, that is the difference between 0.3 MB and 100 MB.

In [ ]:
tracks = torch.randn(500, 128)  # appearance features of 500 tracks

expanded = tracks[:, None, :].expand(-1, 400, -1)  # (500, 400, 128), no copy
repeated = tracks[:, None, :].repeat(1, 400, 1)  # (500, 400, 128), copy
print(expanded.shape, expanded.stride(), f"{expanded.untyped_storage().nbytes() / 1e6:.1f} MB")
print(repeated.shape, repeated.stride(), f"{repeated.untyped_storage().nbytes() / 1e6:.1f} MB")

### Pixel coordinate grids

Many methods need the coordinates of every pixel, e.g. to render a heatmap with a Gaussian peak at every object center, as used by center-based detectors. `torch.meshgrid` creates them. Watch out for the `indexing` argument: with `"ij"`, the first input runs along the rows; with `"xy"`, along the columns. Both return grids of shape `(H, W)` here, but the arguments are swapped. Often, broadcasting `ys[:, None]` and `xs[None, :]` as in section 9.3 is simpler, because it never creates the full grids.

In [ ]:
H, W = 60, 80
ys, xs = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
xs2, ys2 = torch.meshgrid(torch.arange(W), torch.arange(H), indexing="xy")
print(xs.shape, torch.equal(xs, xs2), torch.equal(ys, ys2))

centers = torch.tensor([[20.0, 15.0], [55.0, 40.0], [62.0, 18.0]])  # (x, y) of three objects
sigma = 4.0
dist2 = (xs - centers[:, 0, None, None]) ** 2 + (ys - centers[:, 1, None, None]) ** 2  # (3, H, W)
heatmap = torch.exp(-dist2 / (2 * sigma**2)).amax(dim=0)

plt.imshow(heatmap)
plt.axis("off")
plt.show()

### Reproducibility

Random numbers come from a generator with a *seed*. The same seed gives the same numbers, which is essential to compare two runs of a tracker or to debug a training. PyTorch, NumPy and python's `random` module have separate generators. For a reproducible shuffling, give the `DataLoader` its own seeded `torch.Generator` (next notebook).

In [ ]:
torch.manual_seed(0)
a = torch.rand(3)
torch.manual_seed(0)
b = torch.rand(3)
print(torch.equal(a, b))

## Summary

- Tensors behave like NumPy arrays and convert easily. Watch out for `float32` vs. `float64`, `uint8` overflow and for shared memory (`from_numpy`, slicing views).
- Know your toolbox: indexing with masks and index tensors, `reshape`/`permute`/`unsqueeze`, `cat`/`stack`, `clamp`, reductions with `dim`, `sort`/`topk`/`any`, broadcasting, `@` and `einsum`.
- **Avoid Python loops over tensor elements.** Use vectorized operations and existing library functions. They are orders of magnitude faster and usually shorter and easier to read.
- Compare floats with `torch.allclose`.
- Tasks are solved in `exercise_code` and tested in the notebook.

## References

1. [PyTorch Tutorials](https://pytorch.org/tutorials/)
2. [PyTorch Documentation](https://pytorch.org/docs/stable/index.html)
3. [NumPy Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)